<a href="https://colab.research.google.com/github/soule-geophysics/geol-333-714/blob/main/notebooks/HW1_stairwell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Do this first: File > Save a copy in Drive.**
> You are viewing a shared notebook. Anything you type here is NOT saved.
> Save your own copy now, work only in the copy, and rename it
> `HW1_LASTNAME.ipynb` (exact filename and due date in the assignment
> calendar on Brightspace).

# HW1: Drift Correction and the Free-Air Gradient

**Course:** GEOL 333 / 714, Geophysical Exploration Methods, Fall 2026
**Instructor:** Dax Soule (dax.soule@qc.cuny.edu)
**Due:** Sunday, September 27, 2026, 11:59 PM
**Submit:** This notebook (`.ipynb`) via Brightspace


## What you will do

1. Use Python to compute Earth's gravity from the equation we wrote on the board, the surface form of Newton's law of universal gravitation: `g = GM / r²`.
2. Load a real gravimeter dataset collected in a stairwell.
3. Make two plots: gravity over time (the *drift*) and gravity over elevation (the *free-air trend*).
4. Fit the instrument drift through the repeated ground-level reads, subtract it from every reading, then fit the *drift-corrected* gravity against elevation to measure the free-air gradient. Report it with an uncertainty.
5. Derive the free-air gradient yourself from `g = GM/R²` using the binomial expansion on the Taylor & Binomial card (in the Math Reference Cards PDF on the Resources page).
6. Reflect on which assumptions in `g = GM/R²` this dataset just exposed.

## What you will hand in

This same notebook, with your code, your plots, and your short answers filled in. Save it as `HW1_LASTNAME.ipynb` and upload to Brightspace.


## Loading the data

The code below loads `stairwell.csv` automatically from a stable public web address, so in most cases you do not need to download anything: run the cells.

**If you have no internet, or the link is not live yet,** use the Brightspace fallback:

1. Download `stairwell.csv` from the Brightspace HW1 page.
2. In Google Colab, click the **📁 Files** icon in the left sidebar.
3. Drag `stairwell.csv` into the file panel.
4. In the loading cell below, comment out the `pd.read_csv(DATA_URL)` line and use the commented `pd.read_csv("stairwell.csv")` line instead.

> **Colab deletes uploaded files when the runtime disconnects.**
> If a CSV you uploaded by hand has vanished, re-run the data-loading cell above
> (the URL load restores the data) or re-upload the file. Code and written
> answers persist in your own saved copy.

## Setup

These imports give us NumPy (numbers and arrays), Pandas (tables), and Plotly (interactive plots). All three come pre-installed in Colab; no `pip install` needed.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

# Course accessibility default: colorblind-safe qualitative palette
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe

ModuleNotFoundError: No module named 'plotly'

## Part 1: Newton's Law of Universal Gravitation

From class: the gravitational acceleration at distance `r` from a point mass `M` is

$$g = \frac{G \, M}{r^2}$$

where:
- `G = 6.674e-11 m³ kg⁻¹ s⁻²` (gravitational constant)
- `M = 5.972e24 kg` (mass of Earth)
- `r` = distance from Earth's center (m)

We will treat the Earth as a point mass, the simplest possible model. Run the cell below to compute `g` at sea level and at the top of a hypothetical 100 m building.

**Question 1.1.** Before you run the cell, reason out an estimate. Everything you need is above: `g = GM/r²` (g equals G M over r squared), Earth's radius `R = 6.371e6 m`, and a climb of 100 m.

Work it in three steps and show your arithmetic.

1. What fraction of Earth's radius is 100 m?
2. `g` goes as `1/r²`, so growing `r` by a small fraction costs `g` some fraction. If you are not sure how much, try a number: let `r` grow by 1 per cent, compute `1/1.01²`, and see how far below 1 it lands.
3. Combine them. What fraction of `g` does the climb cost?

Give your estimate as a fraction or a percentage of `g`. A rough number you reasoned to is worth more here than a precise one you did not.

*(Your answer):*

In [ ]:
G = 6.674e-11    # m^3 / (kg s^2)
M_earth = 5.972e24    # kg
R_earth = 6.371e6    # m

g_sea_level = G * M_earth / R_earth**2
g_at_100m = G * M_earth / (R_earth + 100)**2
delta_g = g_sea_level - g_at_100m

print(f'g at sea level:  {g_sea_level:.6f} m/s^2')
print(f'g at 100 m:      {g_at_100m:.6f} m/s^2')
print(f'difference:      {delta_g*1e8:.2f} microgal  (1 microgal = 1e-8 m/s^2)')
print(f'predicted rate:  {delta_g*1e5/100:.4f} mGal per meter   (1 mGal = 1e-5 m/s^2)')

**Question 1.2.** The gravimeter unit milligal (mGal) is the working unit in this assignment. The last line of the printout gives the size of the change in `g` per meter of elevation, without its sign.

1. Report the rate as a signed number in mGal per meter, and say in words which way `g` changes as elevation increases.
2. Compare the printed change with your estimate from 1.1, putting both on the same footing. How close were you, and if you missed, which of your three steps produced the gap?

*(Your answer):*

## Part 2: Load the data

In the stairwell experiment, the operator stood at ground level, took one reading, then walked up to floor 0.5, took another, kept climbing to floor 4.5, then walked back down to ground level and took one final reading. Those two ground-level readings, at the start and the end, bracket the run and let us see how much the instrument's reading *drifted* over the ~68 minutes of work, even though nothing about Earth's gravity actually changed.

The CSV includes a `minutes_since_start` column: the number of minutes elapsed since the first reading. You will use it in Part 4 as the time axis for the drift fit.

## Data source and citation

The dataset you will work with comes from a published teaching collection:

> Parsekian, A. (n.d.). *IGUaNA Unit 3: Gravity and Magnetics Field Data Exercises, Part 3a (Stairwell gravity).* Science Education Resource Center, Carleton College. CC-BY-NC-SA 4.0. [SERC: IGUaNA Unit 3 teaching materials](https://serc.carleton.edu/iguana/teaching_materials/grav_mag/unit3.html)

These are real gravimeter readings taken in a campus stairwell: open-and-close ground-level reads to track instrument drift, plus one reading at each half-floor as the operator climbed.

In [ ]:
# Primary path: load directly from a stable public URL (no upload needed).
DATA_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/stairwell.csv"
df = pd.read_csv(DATA_URL)

# Fallback (no internet, or URL not live yet): download stairwell.csv from the
# Brightspace HW1 page, drag it into the Colab file panel, then comment out the
# two lines above and use:
# df = pd.read_csv("stairwell.csv")

df.head(15)

In [ ]:
df.describe()

In [ ]:
df.info()

**Question 2.1.** Looking at the `station` column: which two rows are the start-of-run and end-of-run ground-level readings? (Hint: one is labeled `Ground level` and the other has a typo. The typo appears in the source data and is preserved here.)

*(Your answer):*

**Question 2.2.** The `reading_mgal` values are around 3423 mGal. The `g_sea_level` you computed in Part 1 was about 9.8 m/s² ≈ 980,000 mGal. Why is the gravimeter showing a number near 3423 instead of near 980,000? *(Hint: a gravimeter is a **relative** instrument. See [relative and absolute measurement](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#relative-and-absolute-measurement) in the glossary.)*

*(Your answer):*

**Question 2.3.** The table carries an `uncertainty_mgal` column: the operator's stated uncertainty on each individual reading.

1. Which station has the largest uncertainty, and which the smallest? How many times larger is one than the other?
2. Suggest one reason a reading taken partway up a stairwell might be less certain than the others. There is more than one defensible answer.
3. This column is the uncertainty on a single reading. In Part 4 you will fit a line through all of them and get an uncertainty on the *slope*. In one sentence, why are those two different things?

*(Your answer):*

## Part 3: Visualize the data

Two plots summarize the dataset.

### 3a. Gravity over time

If we plot the reading against the clock, we expect to see two ground-level readings (start and end) that should be *identical* (same physical place, same Earth) but in practice are slightly different. The difference is the **instrument drift**: spring tension, temperature changes, and tides cause a slow steady creep in the reading.

In [ ]:
fig = px.scatter(
    df,
    x='time',
    y='reading_mgal',
    color='station', symbol='station',
    title='Stairwell gravity readings vs. clock time',
    labels={'time': 'Time (HH:MM:SS)', 'reading_mgal': 'Gravimeter reading (mGal)'},
)
fig.update_traces(marker=dict(size=12))
fig.show()

**Figure description:** A scatter plot with clock time (HH:MM:SS) on the x-axis and gravimeter reading (mGal) on the y-axis; each marker is one station, shown by color. This is the drift view: the start-of-run and end-of-run ground-level reads (the same physical place) sit at the left and right edges, and the small vertical offset between them is the instrument drift that Question 3.1 asks you to read off. Non-visual path: every point's `time`, `reading_mgal`, and `station` are in the data table printed in Part 2, so the two ground-level rows give the drift directly without the plot.

### 3b. Gravity over elevation

Now plot the same readings against the *elevation* of each station above the ground floor. Under `g = GM/R²`, gravity should *decrease* as we move up. The slope of this line, change in g per meter of elevation, is the [**free-air gradient**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#free-air-gradient).

In [ ]:
fig = px.scatter(
    df,
    x='elevation_m',
    y='reading_mgal',
    color='station', symbol='station',
    title='Stairwell gravity readings vs. elevation',
    labels={'elevation_m': 'Elevation above ground (m)', 'reading_mgal': 'Gravimeter reading (mGal)'},
)
fig.update_traces(marker=dict(size=12))
fig.show()

**Figure description:** A scatter plot with elevation above ground (meters) on the x-axis and gravimeter reading (mGal) on the y-axis; each marker is one station, shown by color. This is the free-air view: it shows how the reading changes as the operator climbs the stairwell, which Question 3.2 asks you to describe and compare with the rate from Part 1. Non-visual path: the `elevation_m` and `reading_mgal` columns in the Part 2 data table give every plotted value without the plot.

**Question 3.1.** Look at plot 3a. The two ground-level readings (`Ground level` at the start, `gound level` at the end) sit at nearly the same `y`. Using the two ground-level rows of the Part 2 table, compute by hand how many mGal the reading changed from the start of the run to the end, and report it with its sign. Part 4 Step 1 computes the same number in code; check yours against it when you get there.

*(Your answer):*

**Question 3.2.** Look at plot 3b. These are raw readings, uncorrected, so this is a first look rather than a final answer.

1. Between the ground floor and station 4.5 (15.89 m up), by how many mGal did the reading change? Multiply your Part 1 rate by 15.89 m and compare the two numbers: does the reading fall off as fast as `g = GM/R²` predicts, faster, or more slowly?
2. You showed in 3.1 that the instrument drifted during the run, so part of that change belongs to the instrument rather than to elevation. Before Part 4 corrects it, predict: once the drift is taken out, does the gap between your two numbers widen or narrow? Say why you think so. Part 4 will settle it.

*(Your answer):*

## Part 4: Correct the drift, then measure the free-air gradient

Plot 3b looks like a clean line. **The raw reads still contain drift; correct the drift first.** The instrument drifted the whole time the operator was climbing, so every climbing read carries some drift on top of the real elevation signal. Throwing out the two ground reads does not remove that drift; it only removes the two points that let us *measure* it.

The correct procedure has three steps:

1. **Fit the drift.** The two ground-level reads are at the same physical place (elevation 0), so any difference between them is pure drift. Fit a straight line to `reading_mgal` vs. `minutes_since_start` through *just those two ground reads*. The slope is the instrument's drift rate in mGal per minute.
2. **Subtract the drift from every reading.** Using that drift line, remove `slope × minutes_since_start` from all eleven readings. After this, the two ground reads collapse onto the same value, and every climbing read has had its time-dependent drift removed.
3. **Fit the corrected gravity vs. elevation.** Now the slope of `drift_corrected` vs. `elevation_m` is the free-air gradient, with the drift no longer leaking into it.

We use `numpy.polyfit(x, y, 1)` for the line fits. It returns `[slope, intercept]`.

### Step 1: Fit the drift through the two ground reads

In [ ]:
# Step 1: the two ground-level reads (same place, elevation 0) isolate the drift.
ground = df[df['station'].isin(['Ground level', 'gound level'])]

drift_slope, drift_intercept = np.polyfit(
    ground['minutes_since_start'], ground['reading_mgal'], 1
)

print(f'drift slope:     {drift_slope:+.6f} mGal per minute')
print(f'over the {df["minutes_since_start"].max():.0f}-minute run, total drift: '
      f'{drift_slope * df["minutes_since_start"].max():+.4f} mGal')

### Step 2: Subtract the drift from every reading, then check the closure

Apply the drift line to all eleven readings: `drift_corrected = reading_mgal - drift_slope × minutes_since_start`.

The check that this worked is the **closure**: the two ground-level reads were taken at the same physical place, so once the drift is removed they should give the same number. The cell below prints the closure before and after the correction. Report the after number in Question 4.1.

In [ ]:
# Step 2: subtract the drift line from EVERY reading (not just the climbing ones).
df['drift_corrected'] = df['reading_mgal'] - drift_slope * df['minutes_since_start']

# Confirm the two ground reads now agree (drift removed):
is_ground = df['station'].isin(['Ground level', 'gound level'])
print('ground reads after drift correction:')
print(df[is_ground][['station', 'reading_mgal', 'drift_corrected']].to_string(index=False))

# Closure: the two ground reads are the same physical place, so after the drift is
# removed the difference between them should be zero. This is the number that shows
# Steps 1 and 2 ran; the free-air gradient by itself does not show it.
closure_before = df[is_ground]['reading_mgal'].iloc[-1] - df[is_ground]['reading_mgal'].iloc[0]
closure_after = df[is_ground]['drift_corrected'].iloc[-1] - df[is_ground]['drift_corrected'].iloc[0]

print()
print(f'ground-read closure before correction: {closure_before:+.4f} mGal')
print(f'ground-read closure after correction:  {closure_after:+.4f} mGal')
print('full credit needs the after value within 0.010 mGal of zero.')

### What the fit is doing, and where the uncertainty comes from

Step 3 fits a straight line through nine points that do not lie exactly on one. [**Least squares**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#least-squares)
picks the line that makes the total of the squared vertical misses as small as possible. Each miss
is a [**residual**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#residual): the reading minus what the line says the reading should be at that elevation.

That fit gives a slope, and it also gives a measure of how well nine scattered points pin
that slope down. Two things set it:

- **How much the points scatter** about the line. Tighter scatter, better-pinned slope.
- **How far apart the elevations are.** A slope is a lever: the further apart your two ends, the
  less a given wobble at each end can tilt the line.

`numpy.polyfit(..., cov=True)` returns both the coefficients and a [**covariance matrix**](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#covariance-matrix). The square
root of its first diagonal entry is the **1-sigma uncertainty on the slope**, which is what Step 3
prints after the plus-or-minus. **Sigma is the standard error you met in Week 2**, applied to a
fitted slope instead of to a mean: how far the number would move if you ran the survey again. It is not the uncertainty on any single reading; it is the
uncertainty on the number the whole line produced.

The Matrix Methods Card from Week 2 has the normal equations if you want the algebra underneath.

### Step 3: Fit the drift-corrected gravity vs. elevation

Now that the drift is gone, the slope of `drift_corrected` vs. `elevation_m` is the free-air gradient. We fit it through the climbing reads (the two ground reads are both at elevation 0 and only served to anchor the drift). We also report a **1-sigma uncertainty** on the slope using `numpy.polyfit(..., cov=True)`, so we can say how well the data pin the gradient down, not just a single number.

The figure after the fit is the third required figure: the raw readings, the drift-corrected readings, and the fitted line drawn on top of the points it was fitted to.

In [ ]:
# Step 3: fit the free-air gradient on the DRIFT-CORRECTED gravity.
# Fit through the climbing reads (the ground reads are both at elevation 0 and only
# anchor the drift); report the slope with a 1-sigma uncertainty from the fit.
climb = df[~df['station'].isin(['Ground level', 'gound level'])]

# np.polyfit with cov=True returns the covariance matrix; sqrt of its diagonal is 1 sigma.
coeffs, cov = np.polyfit(
    climb['elevation_m'], climb['drift_corrected'], 1, cov=True
)
gradient, intercept = coeffs
gradient_sigma = np.sqrt(cov[0, 0])

print(f'free-air gradient: {gradient:.4f} +/- {gradient_sigma:.4f} mGal per meter (1 sigma)')
print(f'canonical free-air gradient:                 {-0.3086:.4f} mGal per meter')
print()
# How many sigma is the measured gradient from the canonical value?
n_sigma = abs(gradient - (-0.3086)) / gradient_sigma
print(f'difference from canonical: {n_sigma:.1f} sigma')

In [ ]:
# The drift-corrected fit overlay: raw points, corrected points, and the fitted line.
# Color AND marker symbol both distinguish the two point series (readable in grayscale);
# the fit is a dashed line, so it reads as a line rather than a third set of markers.
plot_df = df.melt(
    id_vars=['station', 'elevation_m'],
    value_vars=['reading_mgal', 'drift_corrected'],
    var_name='series',
    value_name='gravity_mgal',
)
fig = px.scatter(
    plot_df,
    x='elevation_m',
    y='gravity_mgal',
    color='series',
    symbol='series',
    color_discrete_sequence=px.colors.qualitative.Safe,
    title='Drift-corrected gravity vs. elevation, with the fitted free-air gradient',
    labels={
        'elevation_m': 'Elevation above ground (m)',
        'gravity_mgal': 'Gravimeter reading (mGal)',
        'series': 'Series',
    },
)
fig.update_traces(marker=dict(size=11))

# The fitted line from Step 3, drawn across the elevation range of the climbing reads.
elev_line = np.linspace(climb['elevation_m'].min(), climb['elevation_m'].max(), 50)
fig.add_scatter(
    x=elev_line,
    y=gradient * elev_line + intercept,
    mode='lines',
    name=f'fit to drift-corrected: {gradient:.4f} mGal/m',
    line=dict(color=px.colors.qualitative.Safe[3], width=3, dash='dash'),
)
fig.show()

**Figure description:** A scatter plot with elevation above ground (meters, 0 to about 16) on the x-axis and gravimeter reading (mGal) on the y-axis. Three series are shown: the raw `reading_mgal` (circles), the `drift_corrected` reading (diamonds), and the least-squares line fitted to the drift-corrected climbing reads (dashed). The two ground-level points at elevation 0 are separated in the raw series and sit on top of each other in the corrected series, which is the closure from Step 2 seen as a picture. The dashed line is the fit whose slope is the free-air gradient, about -0.287 mGal/m. Non-visual path: the slope, the intercept and the 1-sigma uncertainty are printed by the cell above, and every plotted value is in the `elevation_m`, `reading_mgal` and `drift_corrected` columns of the data table.

**Check before you answer.** Every number the questions below ask for is in the printouts of Steps 1 to 3. If any cell above raised an error, or the ground-read closure after correction in Step 2 is not within 0.010 mGal of zero, use `Runtime → Run all` from the top and read the printouts again.

**Question 4.1.** Step 1 fitted the drift and Step 2 removed it.

1. Report your drift slope in mGal per minute, with its sign. In one sentence, what does that sign mean physically about how the instrument's reading changed across the run?
2. Report the ground-read closure printed in Step 2: the difference between the two ground-level reads after the drift was removed.
3. The drift line was fitted using those same two ground reads and nothing else. Given that, what other value could the closure have come out as? So what does it actually demonstrate, and what does it not demonstrate?

*(Your answer):*

**Question 4.2.** Report your free-air gradient and its 1-sigma uncertainty (Step 3), in the form "value ± uncertainty mGal/m". Step 3 also printed how many of those sigmas separate your gradient from the canonical -0.3086 mGal/m. Under the normal rule, a fitted value lands within 1 sigma of the true value about 68% of the time and within 2 sigma about 95% of the time. Using that rule and the printed separation, decide whether the gap to the canonical value is something the scatter of nine points could produce by chance, or a real difference between the stairwell and the canonical value. One or two sentences.

*(Your answer):*

**Question 4.3.** Part 4 opened with a claim: dropping the two ground reads removes the means of measuring the drift and leaves the drift inside the nine climbing reads. Explain how drift left in the climbing reads ends up in the fitted gradient. The order in which the stations were occupied and the `minutes_since_start` column are the evidence. Estimate how much drift the top station (4.5) carried by the time it was read.

*(Your answer):*

**Question 4.4.** *Forecast.* If we returned to the same building tomorrow at the same time of day and read the gravimeter at ground level, would you expect to get the same number, the start-of-run number, the end-of-run number, or something different? Defend your answer in one or two sentences. (Show your reasoning.)

*(Your answer):*

**Question 4.5.** The stairwell gave you nine readings spread over 15.89 m. Suppose you could go back with time for nine more readings. Which would pin the free-air gradient down better: nine more readings at the same nine elevations, or nine readings spread up a building twice as tall? Say why, using the two things above that set the slope's uncertainty.

*(Your answer):*

## Part 5: Math: derive the free-air gradient

In Part 4 you *measured* a free-air gradient near -0.3 mGal/m. That number comes from `g = GM/R²`, by the binomial expansion on the Taylor & Binomial card (in the Math Reference Cards PDF on the Resources page).

The binomial expansion to first order is

$$(1 + x)^n \approx 1 + n\,x \qquad \text{for small } x.$$

Start from gravity at height $h$ above Earth's surface (radius $R$, mass $M$):

$$g(R+h) = \frac{GM}{(R+h)^2} = \frac{GM}{R^2}\left(1 + \frac{h}{R}\right)^{-2}.$$

You have already used this rule without its name. In Question 1.1 you computed $1/1.01^2$ and found 0.9803; the rule says $1 + (-2)(0.01) = 0.98$. That is the same arithmetic.

The exact expression $GM/(R+h)^2$ is perfectly computable, so the approximation is not here to save effort. It is here because it shows the **shape**: gravity falls off *linearly* with height near the surface. That linearity is the reason a free-air correction can be one constant times an elevation, and the reason the Bouguer correction you meet next week has the same form.

Here the small parameter is $x = h/R$ (a 100 m building over a 6371 km Earth is $x \approx 1.6\times10^{-5}$, very small), and the exponent is $n = -2$. Applying the expansion:

$$g(R+h) \approx \frac{GM}{R^2}\left(1 - 2\,\frac{h}{R}\right) = g_0\left(1 - \frac{2h}{R}\right),$$

where $g_0 = GM/R^2$ is the surface value. So gravity falls off *linearly* with height near the surface, and the slope (the free-air gradient) is

$$\frac{dg}{dh} \approx -\frac{2\,g_0}{R}.$$

Run the cell below to plug in the numbers and compare to the canonical -0.3086 mGal/m.

In [ ]:
G = 6.674e-11    # m^3 / (kg s^2)
M_earth = 5.972e24    # kg
R_earth = 6.371e6    # m

g0 = G * M_earth / R_earth**2
free_air_gradient = -2 * g0 / R_earth        # m/s^2 per meter of elevation
free_air_gradient_mgal = free_air_gradient * 1e5    # 1 mGal = 1e-5 m/s^2

print(f'g0 (surface gravity):         {g0:.4f} m/s^2')
print(f'free-air gradient:            {free_air_gradient:.4e} m/s^2 per meter')
print(f'free-air gradient (mGal/m):   {free_air_gradient_mgal:.4f}')
print()
print(f'canonical value:              {-0.3086:.4f} mGal/m')

**Question 5.1.** The derivation named the small parameter, $x = h/R$, and gave its size for a 100 m building. The Taylor & Binomial card gives the next term of the expansion, $\tfrac{n(n-1)}{2}x^2$. Write that term for $n = -2$, then compare its size to the first-order term $2x$ at the top of the stairwell, $h = 15.89$ m. In one sentence, why is the first-order term enough here?

*(Your answer):*

**Question 5.2.** Part 4 measured the gradient inside the stairwell, about -0.287 ± 0.008 mGal/m. This part derived about -0.308 mGal/m from $g = GM/R^2$ alone, with no field data. Which one is the "true" free-air gradient? Then give one physical reason the gradient measured inside a building comes out smaller in magnitude than the derived one, and say which way that cause pushes the readings as the operator climbs. (Hint: think about what mass is between the gravimeter and open air.)

*(Your answer):*

### Why first order is enough, and where it stops being enough

Keeping the second-order term makes the magnitude of the gradient shrink a little as you climb,
because you are further from Earth's centre and the falloff is gentler. Across the stairwell that
comes to 0.308256 mGal/m at the ground against 0.308254 at the top, four decimal places below the
plus-or-minus 0.008 you measured in Part 4.

The fractional shift is about $3h/R$, so it would take a height of roughly **55 km** before the
curvature moved the gradient by as much as your own 1-sigma uncertainty. That is sixty-six times the
tallest building on Earth and five times the cruising altitude of an airliner. **The first-order
free-air correction is safe for any survey you are likely to run**, and it stays safe well past
anything you could climb.


## Part 6: Reflection

From Newton's law of universal gravitation, gravity at a distance r from Earth's center is

$$g = \frac{G \, M}{r^2}$$

It assumes Earth is a point mass, the observer is at the reference surface, there's nothing between the observer and the reference, and the instrument is stable in time. Each of those assumptions becomes a *correction* later in this course.

**Question 6.1.** Question 5.2 already placed one of the four: the building's mass sits between the observer and the reference, so the third assumption is broken, and the 0.022 mGal/m gap is the size of the break. For the other three: which did the stairwell dataset *break*, and which did it leave untested? Cite the specific feature of the data behind each answer.

*(Your answer):*

**Question 6.2.** In 4.1 you worked out why the closure had to come out the way it did. Suppose the operator had taken a third ground-level reading midway through the run, near the 30-minute mark. What could a drift fit through three ground reads tell you that the fit through two cannot? One or two sentences.

*(Your answer):*

**Question 6.3.** *(Optional, ungraded)* Which result in this dataset did you least expect, and what would you check to test it?

*(Your answer):*


## How to submit

1. Run all cells from the top. (`Runtime → Run all` in Colab.)
2. Make sure all your short answers are filled in.
3. `File → Download → Download .ipynb`.
4. Rename the file to `HW1_LASTNAME.ipynb` (e.g., `HW1_Smith.ipynb`).
5. Upload to the Brightspace HW1 dropbox by **Sunday September 27, 11:59 PM**.

If something is broken or unclear, post on the **Ask the Class (General Q&A)** discussion topic. Other students may have the same question.